# Molecular Graph `nbits` Benchmark

This notebook loads the MoleculeNet HIV dataset through `NSPPK.load_from(...)` and studies how predictive performance changes as `nbits` varies.

The train/test split is created once and reused for every `nbits` value, so only the feature hashing dimension changes across runs.


In [ ]:
from pathlib import Path
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from rdkit import RDLogger
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from statsmodels.nonparametric.smoothers_lowess import lowess

REPO_CANDIDATES = [Path.cwd().resolve(), Path.cwd().resolve().parent]
REPO_ROOT = next(candidate for candidate in REPO_CANDIDATES if (candidate / 'src').exists())
SRC_DIR = REPO_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from nsppk import NSPPK

DATASET_URL = 'https://deepchemdata.s3-us-west-1.amazonaws.com/datasets/HIV.csv'

NBITS_VALUES = list(range(6, 20))
RADIUS = 1
DISTANCE = 4
CONNECTOR = 1
LIMIT = 8000
BALANCE_DATASET = True
TRAIN_FRACTION = 0.7
TEST_SIZE = 1 - TRAIN_FRACTION
RANDOM_STATE = 42

RDLogger.DisableLog('rdApp.*')


## Load molecular graphs

The notebook uses `NSPPK.load_from(...)` to parse the dataset. The HIV activity label is recovered from the second CSV column, which the built-in SMILES reader stores in `graph.graph['name']`.


In [ ]:
loader = NSPPK(radius=RADIUS, distance=DISTANCE, connector=CONNECTOR, nbits=max(NBITS_VALUES), parallel=True)
graphs = loader.load_from(
    DATASET_URL,
    'smiles',
    limit=LIMIT,
    random_state=RANDOM_STATE,
    balance=BALANCE_DATASET,
    label_extractor=lambda graph: int(graph.graph['name']),
)
labels = np.asarray([int(graph.graph['name']) for graph in graphs])

original_dataset_size = len(loader.load_from(DATASET_URL, 'smiles'))

print('original loaded molecules:', original_dataset_size)
print('balance dataset:', BALANCE_DATASET)
print('molecules after balance:', len(graphs))
print('molecules after balance/limit:', len(graphs))
print(f'positive rate: {labels.mean():.3f}')
print('class counts:', {0: int((labels == 0).sum()), 1: int((labels == 1).sum())})
print('example node label:', next(iter(graphs[0].nodes(data=True)))[1]['label'])
print('example edge label:', next(iter(graphs[0].edges(data=True)))[2]['label'])


## Fixed train/test split

If `BALANCE_DATASET = True`, we first build the largest balanced subset possible using all examples from the minority class and the same number from the majority class. If `LIMIT` is larger than that balanced size, we then add extra examples from the majority class until we reach `LIMIT` or exhaust the dataset. We finally create one fixed 70% train / 30% test split that is reused for every `nbits` value.


In [ ]:
indices = np.arange(len(graphs))
train_idx, test_idx = train_test_split(
    indices,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=labels,
)

train_graphs = [graphs[i] for i in train_idx]
test_graphs = [graphs[i] for i in test_idx]
y_train = labels[train_idx]
y_test = labels[test_idx]

print('limit:', LIMIT)
print('balance dataset:', BALANCE_DATASET)
print('train fraction:', TRAIN_FRACTION)
print('train size:', len(train_graphs))
print('test size:', len(test_graphs))
print(f'train positive rate: {y_train.mean():.2f}')
print(f'test positive rate: {y_test.mean():.2f}')


## Sweep `nbits` on the same training and held-out test sets

We keep `(radius, distance, connector)` fixed and retrain a scikit-learn pipeline for every `nbits` value.


In [ ]:
results = []
n_runs = len(NBITS_VALUES)

print(
    f"{'run':>7} | {'nbits':>5} | {'features':>8} | {'nnz/graph':>9} | "
    f"{'memory (MB)':>11} | {'ROC-AUC':>7} | {'AP':>4} | {'pipeline':>9}"
)
print('-' * 78)

for run_idx, nbits in enumerate(NBITS_VALUES, start=1):
    pipeline = Pipeline([
        ('nsppk', NSPPK(
            radius=RADIUS,
            distance=DISTANCE,
            connector=CONNECTOR,
            nbits=nbits,
            dense=False,
            parallel=True,
        )),
        ('classifier', SGDClassifier(
            loss='hinge',
            penalty='l2',
            alpha=1e-4,
            class_weight='balanced',
            max_iter=2000,
            tol=1e-3,
            random_state=RANDOM_STATE,
        )),
    ])

    t0 = time.time()
    pipeline.fit(train_graphs, y_train)
    pipeline_time = time.time() - t0

    vectorizer = pipeline.named_steps['nsppk']
    classifier = pipeline.named_steps['classifier']
    X_train = vectorizer.transform(train_graphs)
    X_test = vectorizer.transform(test_graphs)

    train_memory_size_mb = (X_train.data.nbytes + X_train.indices.nbytes + X_train.indptr.nbytes) / (1024 ** 2)
    test_scores = classifier.decision_function(X_test)
    test_roc_auc = roc_auc_score(y_test, test_scores)
    test_average_precision = average_precision_score(y_test, test_scores)

    result = {
        'nbits': nbits,
        'n_features': X_train.shape[1],
        'train_nnz_mean': float(X_train.getnnz(axis=1).mean()),
        'train_memory_size_mb': train_memory_size_mb,
        'test_roc_auc': test_roc_auc,
        'test_average_precision': test_average_precision,
        'pipeline_time_sec': pipeline_time,
    }
    results.append(result)
    print(
        f"[{run_idx:>2d}/{n_runs:<2d}] | "
        f"{nbits:>5d} | "
        f"{result['n_features']:>8d} | "
        f"{result['train_nnz_mean']:>9.2f} | "
        f"{train_memory_size_mb:>11.2f} | "
        f"{test_roc_auc:>7.2f} | "
        f"{test_average_precision:>4.2f} | "
        f"{pipeline_time:>8.2f}s"
    )

results_df = pd.DataFrame(results)
results_df.round(2)


In [ ]:
nbits_values = results_df['nbits'].to_numpy()
x_ticks = np.arange(int(nbits_values.min()), int(nbits_values.max()) + 1)
roc_auc_values = results_df['test_roc_auc'].to_numpy()
ap_values = results_df['test_average_precision'].to_numpy()
pipeline_values = results_df['pipeline_time_sec'].to_numpy()
roc_auc_loess = lowess(roc_auc_values, nbits_values, frac=0.75, return_sorted=True)
ap_loess = lowess(ap_values, nbits_values, frac=0.75, return_sorted=True)
pipeline_loess = lowess(pipeline_values, nbits_values, frac=0.85, return_sorted=True)

pdf_dir = Path('.')
saved_pdfs = []

def finalize_plot(fig, axes, filename):
    for ax in axes:
        ax.set_xticks(x_ticks)
        ax.set_xlim(x_ticks[0], x_ticks[-1])
    fig.tight_layout()
    pdf_path = pdf_dir / filename
    fig.savefig(pdf_path, format='pdf', bbox_inches='tight')
    saved_pdfs.append(pdf_path)
    plt.show()

fig, ax = plt.subplots(figsize=(6, 4))
roc_auc_line, = ax.plot(nbits_values, roc_auc_values, marker='o', alpha=0.5, label='ROC-AUC')
ap_line, = ax.plot(nbits_values, ap_values, marker='o', alpha=0.5, label='Average precision')
ax.plot(roc_auc_loess[:, 0], roc_auc_loess[:, 1], linestyle='-', linewidth=4, alpha=1.0, color=roc_auc_line.get_color())
ax.plot(ap_loess[:, 0], ap_loess[:, 1], linestyle='-', linewidth=4, alpha=1.0, color=ap_line.get_color())
ax.set_xlabel('nbits')
ax.set_ylabel('Test performance')
ax.set_title('Predictive performance vs nbits')
ax.grid(True, alpha=0.3)
ax.legend()
finalize_plot(fig, [ax], 'nsppk_molecular_graph_predictive_performance_vs_nbits.pdf')

fig, ax = plt.subplots(figsize=(6, 4))
runtime_line, = ax.plot(nbits_values, pipeline_values, marker='o', color='tab:blue', alpha=0.5, label='Pipeline time')
ax.plot(pipeline_loess[:, 0], pipeline_loess[:, 1], linestyle='-', linewidth=4, alpha=1.0, color=runtime_line.get_color())
ax.set_xlabel('nbits')
ax.set_ylabel('Seconds')
ax.set_title('Pipeline runtime vs nbits')
ax.grid(True, alpha=0.3)
ax.legend()
finalize_plot(fig, [ax], 'nsppk_molecular_graph_runtime_vs_nbits.pdf')

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(nbits_values, results_df['train_nnz_mean'], marker='o', color='tab:green')
ax.set_xlabel('nbits')
ax.set_ylabel('Mean nonzeros per training graph')
ax.set_title('Sparsity vs nbits')
ax.grid(True, alpha=0.3)
finalize_plot(fig, [ax], 'nsppk_molecular_graph_sparsity_vs_nbits.pdf')

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(nbits_values, results_df['train_memory_size_mb'], marker='o', color='tab:purple')
ax.set_xlabel('nbits')
ax.set_ylabel('Training matrix size (MB)')
ax.set_title('Memory footprint vs nbits')
ax.grid(True, alpha=0.3)
finalize_plot(fig, [ax], 'nsppk_molecular_graph_memory_vs_nbits.pdf')

print('Saved PDF files:')
for pdf_path in saved_pdfs:
    print(f'- {pdf_path.name}')
